In [1]:
from genominterv.remapping import remap
from genominterv.remapping import interval_distance, genomic
from genominterv.remapping import remap_interval_data


In [2]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("/home/johanulstrup/johan_gpn/people/johanulsrup/johan_gpn/data/human/autosome_hg38_compartments.csv")

# Check the first few rows to verify it loaded correctly
print(df.head())

# Filter for chromosome 8 and chromosome X
chr8_df = df[df["chrom"] == "chr8"].reset_index(drop=True)


# Optional: display the number of rows for each
print(f"chr8 rows: {len(chr8_df)}")


print(chr8_df.head())



  chrom   start     end  E1_X compart_X  E1_Y compart_Y
0  chr1       0   50000   NaN         B   NaN         B
1  chr1   50000  100000   NaN         B   NaN         B
2  chr1  100000  150000   NaN         B   NaN         B
3  chr1  150000  200000   NaN         B   NaN         B
4  chr1  200000  250000   NaN         B   NaN         B
chr8 rows: 2903
  chrom   start     end      E1_X compart_X     E1_Y compart_Y
0  chr8       0   50000       NaN         B      NaN         B
1  chr8   50000  100000       NaN         B      NaN         B
2  chr8  100000  150000       NaN         B      NaN         B
3  chr8  150000  200000       NaN         B      NaN         B
4  chr8  200000  250000 -0.084579         B -0.29916         B


In [3]:
import pandas as pd

# Load the CSV file
chrx_df = pd.read_csv("/home/johanulstrup/johan_gpn/people/johanulsrup/johan_gpn/data/human/chrX_hg38_compartments.csv")

# Optional: display the number of rows for each
print(f"chr8 rows: {len(chr8_df)}")


print(chrx_df.head())

chr8 rows: 2903
  chrom   start     end  E1 compart
0  chrX       0   50000 NaN       B
1  chrX   50000  100000 NaN       B
2  chrX  100000  150000 NaN       B
3  chrX  150000  200000 NaN       B
4  chrX  200000  250000 NaN       B


In [4]:
import pandas as pd

# assuming chr8_df is already loaded

# select the needed columns and rename
df_chr8 = chr8_df[["compart_X", "start", "end", "chrom"]].rename(
    columns={"compart_X": "comp"}
)
print("chr8 data:")
print(df_chr8.head())
print(df_chr8.shape)

# select the needed columns and rename
df_chrx = chrx_df[["compart", "start", "end", "chrom"]].rename(
    columns={"compart": "comp"}
)
print("chrX data:")
print(df_chrx.head())
print(df_chrx.shape)


chr8 data:
  comp   start     end chrom
0    B       0   50000  chr8
1    B   50000  100000  chr8
2    B  100000  150000  chr8
3    B  150000  200000  chr8
4    B  200000  250000  chr8
(2903, 4)
chrX data:
  comp   start     end chrom
0    B       0   50000  chrX
1    B   50000  100000  chrX
2    B  100000  150000  chrX
3    B  150000  200000  chrX
4    B  200000  250000  chrX
(3121, 4)


In [12]:
import pandas as pd

def create_A_B_compartments(df):
    """
    Merge consecutive A/B compartments into continuous intervals.
    
    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: ["comp", "start", "end", "chrom"]

    Returns
    -------
    merged_df : pd.DataFrame
        DataFrame with merged intervals: ["comp", "start", "end", "chrom"]
    """
    # Ensure proper sorting
    df = df.sort_values(by=["chrom", "start"]).reset_index(drop=True)
    
    merged = []
    current_comp = df.loc[0, "comp"]
    current_start = df.loc[0, "start"]
    current_chrom = df.loc[0, "chrom"]

    for i in range(1, len(df)):
        row = df.loc[i]
        if row["comp"] != current_comp or row["chrom"] != current_chrom:
            # Close previous interval
            merged.append([current_comp, current_start, df.loc[i-1, "end"], current_chrom])
            # Start new one
            current_comp = row["comp"]
            current_start = row["start"]
            current_chrom = row["chrom"]

    # Add last interval
    merged.append([current_comp, current_start, df.loc[len(df)-1, "end"], current_chrom])

    return pd.DataFrame(merged, columns=["comp", "start", "end", "chrom"])

df_chrx_intervals = create_A_B_compartments(df_chrx)
df_chr8_intervals = create_A_B_compartments(df_chr8)

print(df_chrx_intervals.head())
print(df_chr8_intervals.head())


  comp    start      end chrom
0    B        0  2800000  chrX
1    A  2800000  3800000  chrX
2    B  3800000  3950000  chrX
3    A  3950000  4550000  chrX
4    B  4550000  4600000  chrX
  comp    start       end chrom
0    B        0   1750000  chr8
1    A  1750000   1800000  chr8
2    B  1800000   9750000  chr8
3    A  9750000   9800000  chr8
4    B  9800000  10300000  chr8


In [5]:
# import pandas as pd

# bed_file = "/home/johanulstrup/johan_gpn/people/johanulsrup/johan_gpn/data/human/gnomad_v2.1_sv.sites.bed"

# df = pd.read_csv(bed_file, sep="\t", header=None, comment="#")
# print("Number of columns:", df.shape[1])
# print(df.head())


In [6]:
import pandas as pd

# Path to the BEDGraph file
recomb_file = "/home/johanulstrup/johan_gpn/people/johanulsrup/johan_gpn/data/human/recombMat.bedGraph"

# Load the BEDGraph file (tab-delimited, no header by default)
recomb_df = pd.read_csv(recomb_file, sep="\t", header=None, names=["chrom", "start", "end", "value"])

recomb_df = recomb_df[["chrom","start", "end"]]
# Separate data for chr8
chr8_recomb = recomb_df[recomb_df["chrom"] == "chr8"].reset_index(drop=True)

# Separate data for chrX
chrx_recomb = recomb_df[recomb_df["chrom"] == "chrX"].reset_index(drop=True)

# Optional: Check the number of entries
print(f"chr8 rows: {len(chr8_recomb)}")
print(chr8_recomb.head())
print(f"chrX rows: {len(chrx_recomb)}")
print(chrx_recomb.head())



chr8 rows: 56890
  chrom   start     end
0  chr8  727305  757884
1  chr8  757884  763377
2  chr8  763377  764383
3  chr8  764383  766867
4  chr8  766867  769043
chrX rows: 23976
  chrom    start      end
0  chrX  3532526  3533229
1  chrX  3533229  3534231
2  chrX  3534231  3580516
3  chrX  3580516  3582267
4  chrX  3582267  3582957


In [13]:
# Function to clean, cast, and sort genomic intervals
def clean_and_sort(df):
    df_clean = df.dropna(subset=["start", "end"]).copy()
    df_clean["start"] = df_clean["start"].astype(int)
    df_clean["end"] = df_clean["end"].astype(int)
    df_clean = df_clean.sort_values(by=["chrom", "start", "end"]).reset_index(drop=True)
    return df_clean

# Apply to recombination data
chrx_recomb_clean = clean_and_sort(chrx_recomb)
chr8_recomb_clean = clean_and_sort(chr8_recomb)

# Apply to compartment anchor data
df_chrx_clean = clean_and_sort(df_chrx_intervals)
df_chr8_clean = clean_and_sort(df_chr8_intervals)

# Quick peek
print("chrX recomb:\n", chrx_recomb_clean.head(), "\n")
print("chrX compartments:\n", df_chrx_clean.head())

chrX recomb:
   chrom    start      end
0  chrX  3532526  3533229
1  chrX  3533229  3534231
2  chrX  3534231  3580516
3  chrX  3580516  3582267
4  chrX  3582267  3582957 

chrX compartments:
   comp    start      end chrom
0    B        0  2800000  chrX
1    A  2800000  3800000  chrX
2    B  3800000  3950000  chrX
3    A  3950000  4550000  chrX
4    B  4550000  4600000  chrX


In [14]:
# Remap chrX compartments relative to recombination intervals
remapped_chrx = remap_interval_data(chrx_recomb_clean, df_chrx_clean, include_prox_coord=True)
remapped_chrx_int = remap_interval_data( df_chrx_clean, chrx_recomb_clean, include_prox_coord=True)
print("Remapped chrX data:")
print(remapped_chrx.shape)
print(remapped_chrx.head())
print("Remapped invertchrX data:")
print(remapped_chrx_int.shape)
print(remapped_chrx_int.head())

Remapped chrX data:
(47952, 7)
   start  end  start_prox  end_prox chrom  start_orig  end_orig
0    NaN  NaN         NaN       NaN  chrX     3532526   3533229
1    NaN  NaN         NaN       NaN  chrX     3532526   3533229
2    NaN  NaN         NaN       NaN  chrX     3533229   3534231
3    NaN  NaN         NaN       NaN  chrX     3533229   3534231
4    NaN  NaN         NaN       NaN  chrX     3534231   3580516
Remapped invertchrX data:
(638, 8)
      start        end  start_prox  end_prox comp  start_orig  end_orig chrom
0 -732526.0 -3532526.0         NaN       NaN    B           0   2800000  chrX
1       NaN        NaN         NaN       NaN    A     2800000   3800000  chrX
2       NaN        NaN         NaN       NaN    A     2800000   3800000  chrX
3       NaN        NaN         NaN       NaN    B     3800000   3950000  chrX
4       NaN        NaN         NaN       NaN    B     3800000   3950000  chrX


In [9]:
# Remap chr8 compartments relative to recombination intervals
remapped_chr8 = remap_interval_data(chr8_recomb, df_chr8, include_prox_coord=True)
print("Remapped chr8 data:")
print(remapped_chr8.shape)

remapped_chr8.head()


Remapped chr8 data:
(113780, 7)


,start,end,start_prox,end_prox,chrom,start_orig,end_orig
0,NaN,NaN,NaN,NaN,chr8,727305,757884
1,NaN,NaN,NaN,NaN,chr8,727305,757884
2,NaN,NaN,NaN,NaN,chr8,757884,763377
3,NaN,NaN,NaN,NaN,chr8,757884,763377
4,NaN,NaN,NaN,NaN,chr8,763377,764383
